# Reviewer #1, Q3 — Statistical diagnostics for the RF / RF-SHAP model

Computes, for all 8 years, using **your exact GridSearchCV grid**:

1. **VIF** (multicollinearity) → `VIF_results.csv`
2. **Random 5-fold CV** (your original workflow)
3. **Spatial-block 5-fold CV** (honest, no spatial leakage)
4. **RF uncertainty**: R²/RMSE/MAE mean ± SD across folds
5. **Best hyperparameters** per year → `RF_best_params.csv`

**Before running:** set `HERE` in the next cell to the folder containing `1990.csv … 2025.csv`.
If this notebook is already in the same folder as the CSVs, leave it as `""`.

Uses all CPU cores (`n_jobs=-1`). Expect ~30–90 min. Each year is checkpointed, so
progress is never lost. Just run the cells top to bottom.

## 1. Settings — edit `HERE` if needed

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ---- EDIT THIS if your CSVs are in a different folder ----
# "" means: same folder as this notebook.
# Example (Windows): HERE = r"C:\\Users\\you\\Desktop\\revision"
# Example (Mac/Linux): HERE = "/home/you/revision"
HERE = ""

YEARS   = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]
PREDS   = ["NDVI", "NDBI", "NDWI", "SAVI"]
BLOCK_M = 5000.0     # 5 km spatial blocks (grid is 500 m; n=14419)
NFOLDS  = 5
SEED    = 42

# YOUR EXACT GRID (from your RF-SHAP script)
PARAM_GRID = {
    "n_estimators":     [200, 500],
    "max_depth":        [10, 20],
    "min_samples_split":[2, 5],
    "min_samples_leaf": [1, 2],
    "max_features":     ["sqrt"],
}
print("Settings loaded. HERE =", repr(HERE) or "(current folder)")

## 2. Helper functions

In [ ]:
def load_year(y):
    """Auto-detect columns for a given year; return X, y, coords (drops NaN / -9999)."""
    df = pd.read_csv(os.path.join(HERE, f"{y}.csv"))
    colmap = {}
    for p in PREDS + ["LST"]:
        hit = [c for c in df.columns if c.upper() == f"{p}_{y}".upper()]
        if not hit:
            raise ValueError(f"{y}.csv is missing column {p}_{y}")
        colmap[p] = hit[0]
    xcol = [c for c in df.columns if c.upper() == "X_COR"][0]
    ycol = [c for c in df.columns if c.upper() == "Y_COR"][0]
    keep = [xcol, ycol] + [colmap[p] for p in PREDS] + [colmap["LST"]]
    d = df[keep].apply(pd.to_numeric, errors="coerce").replace([-9999, -9999.0], np.nan).dropna()
    X  = d[[colmap[p] for p in PREDS]].values
    yv = d[colmap["LST"]].values
    xy = d[[xcol, ycol]].values
    return X, yv, xy

def spatial_blocks(xy, block=BLOCK_M):
    bx = np.floor((xy[:, 0] - xy[:, 0].min()) / block).astype(int)
    by = np.floor((xy[:, 1] - xy[:, 1].min()) / block).astype(int)
    return bx * 100000 + by

def vif_for(X):
    out = {}
    for i, p in enumerate(PREDS):
        others = np.delete(X, i, axis=1)
        r2 = LinearRegression().fit(others, X[:, i]).score(others, X[:, i])
        out[p] = np.inf if r2 >= 1 else 1.0 / (1.0 - r2)
    return out

# quick check that files load
try:
    _X, _y, _xy = load_year(YEARS[0])
    print(f"OK: {YEARS[0]}.csv loaded  ->  {_X.shape[0]} pixels, "
          f"{len(set(spatial_blocks(_xy)))} spatial blocks")
except Exception as e:
    print("PROBLEM loading files -> check HERE path. Error:", e)

## 3. Run all diagnostics

This is the long cell (~30–90 min). It prints results per year as it goes and
saves `VIF_results.csv`, `RF_CV_comparison.csv`, `RF_best_params.csv` after **each**
year, so nothing is lost if interrupted.

In [ ]:
vif_rows, cv_rows, param_rows = [], [], []
t_all = time.time()

for y in YEARS:
    t0 = time.time()
    X, yv, grp_xy = load_year(y)
    grp = spatial_blocks(grp_xy)
    n, nb = len(yv), len(np.unique(grp))
    print(f"\n=== {y}  (n={n}, spatial blocks={nb}) ===", flush=True)

    # (1) VIF
    v = vif_for(X)
    vif_rows.append({"Year": y, "n": n, **{f"VIF_{p}": round(v[p], 2) for p in PREDS}})
    print("  VIF:", {p: round(v[p], 1) for p in PREDS}, flush=True)

    # (2) RANDOM 5-fold CV (replicates your original workflow)
    kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    r_r2, r_rmse, r_mae = [], [], []
    for tr, te in kf.split(X):
        gs = GridSearchCV(RandomForestRegressor(random_state=SEED),
                          PARAM_GRID, scoring="r2", cv=5, n_jobs=-1)
        gs.fit(X[tr], yv[tr])
        p = gs.best_estimator_.predict(X[te])
        r_r2.append(r2_score(yv[te], p))
        r_rmse.append(np.sqrt(mean_squared_error(yv[te], p)))
        r_mae.append(mean_absolute_error(yv[te], p))
    print(f"  Random  CV: R2={np.mean(r_r2):.3f}\u00b1{np.std(r_r2):.3f} "
          f"RMSE={np.mean(r_rmse):.3f} MAE={np.mean(r_mae):.3f}", flush=True)

    # (3) NESTED SPATIAL-BLOCK CV (no spatial leakage in tuning or evaluation)
    outer = GroupKFold(n_splits=NFOLDS)
    s_r2, s_rmse, s_mae, chosen = [], [], [], []
    for tr, te in outer.split(X, yv, groups=grp):
        inner = GroupKFold(n_splits=NFOLDS)
        gs = GridSearchCV(RandomForestRegressor(random_state=SEED), PARAM_GRID,
                          scoring="r2", n_jobs=-1,
                          cv=inner.split(X[tr], yv[tr], groups=grp[tr]))
        gs.fit(X[tr], yv[tr])
        p = gs.best_estimator_.predict(X[te])
        s_r2.append(r2_score(yv[te], p))
        s_rmse.append(np.sqrt(mean_squared_error(yv[te], p)))
        s_mae.append(mean_absolute_error(yv[te], p))
        chosen.append(gs.best_params_)
    print(f"  Spatial CV: R2={np.mean(s_r2):.3f}\u00b1{np.std(s_r2):.3f} "
          f"RMSE={np.mean(s_rmse):.3f}\u00b1{np.std(s_rmse):.3f} "
          f"MAE={np.mean(s_mae):.3f}\u00b1{np.std(s_mae):.3f}", flush=True)
    print(f"  R2 drop (random - spatial) = {np.mean(r_r2)-np.mean(s_r2):.3f}", flush=True)

    # (4) Full-data best params (the model you would report for year y)
    gs_full = GridSearchCV(RandomForestRegressor(random_state=SEED),
                           PARAM_GRID, scoring="r2", cv=5, n_jobs=-1)
    gs_full.fit(X, yv)
    param_rows.append({"Year": y, **gs_full.best_params_})
    print("  Best params (full data):", gs_full.best_params_, flush=True)

    cv_rows.append({
        "Year": y, "n": n, "spatial_blocks": nb,
        "rnd_R2": np.mean(r_r2),  "rnd_R2_sd": np.std(r_r2),
        "rnd_RMSE": np.mean(r_rmse), "rnd_MAE": np.mean(r_mae),
        "sp_R2": np.mean(s_r2),   "sp_R2_sd": np.std(s_r2),
        "sp_RMSE": np.mean(s_rmse), "sp_RMSE_sd": np.std(s_rmse),
        "sp_MAE": np.mean(s_mae),   "sp_MAE_sd": np.std(s_mae),
        "R2_drop": np.mean(r_r2) - np.mean(s_r2),
    })

    # checkpoint after every year
    pd.DataFrame(vif_rows).to_csv(os.path.join(HERE, "VIF_results.csv"), index=False)
    pd.DataFrame(cv_rows).round(4).to_csv(os.path.join(HERE, "RF_CV_comparison.csv"), index=False)
    pd.DataFrame(param_rows).to_csv(os.path.join(HERE, "RF_best_params.csv"), index=False)
    json.dump(chosen, open(os.path.join(HERE, f"spatial_params_{y}.json"), "w"), indent=2)
    print(f"  [checkpoint saved] year took {(time.time()-t0)/60:.1f} min", flush=True)

print(f"\nALL DONE in {(time.time()-t_all)/60:.1f} min")
print("Outputs: VIF_results.csv, RF_CV_comparison.csv, RF_best_params.csv")

## 4. Preview the results

In [ ]:
print("VIF_results.csv");            display(pd.read_csv(os.path.join(HERE,"VIF_results.csv")))
print("\nRF_CV_comparison.csv");       display(pd.read_csv(os.path.join(HERE,"RF_CV_comparison.csv")))
print("\nRF_best_params.csv");         display(pd.read_csv(os.path.join(HERE,"RF_best_params.csv")))